In [ ]:

# 1) Build CRITIC index (lnCO2 + SO2 + Dust + Wastewater) and save files
import sys
import numpy as np
import pandas as pd

if hasattr(sys.stdout, 'reconfigure'):
    sys.stdout.reconfigure(encoding='utf-8')
if hasattr(sys.stderr, 'reconfigure'):
    sys.stderr.reconfigure(encoding='utf-8')

pd.set_option('display.max_columns', None)

# Read source panel
df = pd.read_csv('final_data_rebuilt_official_v2_utf8.csv', encoding='utf-8-sig')

# Use column positions to avoid encoding issues with column names
city = df.iloc[:, 0].astype(str)
city_code = df.iloc[:, 1]
year = pd.to_numeric(df.iloc[:, 2], errors='coerce')
treat = pd.to_numeric(df.iloc[:, 3], errors='coerce')
batch = df.iloc[:, 4]

co2 = pd.to_numeric(df.iloc[:, 5], errors='coerce')
so2 = pd.to_numeric(df.iloc[:, 6], errors='coerce')
dust = pd.to_numeric(df.iloc[:, 7], errors='coerce')
water = pd.to_numeric(df.iloc[:, 8], errors='coerce')

work = pd.DataFrame({
    'city': city,
    'city_code': city_code,
    'year': year,
    'treat': treat,
    'batch': batch,
    'lnco2': np.log(co2.where(co2 > 0)),
    'so2': so2,
    'dust': dust,
    'water': water,
}).replace([np.inf, -np.inf], np.nan)


def build_critic(df_in, cols):
    Z = pd.DataFrame(index=df_in.index)
    for c in cols:
        s = pd.to_numeric(df_in[c], errors='coerce')
        mn, mx = s.min(skipna=True), s.max(skipna=True)
        Z[c] = np.nan if (pd.isna(mn) or pd.isna(mx) or mx == mn) else (s - mn) / (mx - mn)

    Zf = Z.copy()
    for c in cols:
        med = Zf[c].median(skipna=True)
        Zf[c] = Zf[c].fillna(0.0 if pd.isna(med) else med)

    std = Zf.std(ddof=1)
    corr = Zf.corr().fillna(0)
    C = pd.Series({c: std[c] * (1 - corr.loc[c]).sum() for c in cols})
    w = C / C.sum() if (C.sum() != 0 and not C.isna().all()) else pd.Series([1 / len(cols)] * len(cols), index=cols)
    score = (Zf * w).sum(axis=1)
    return score, w

work['pollution_index_critic_lnco2_3poll'], critic_weights = build_critic(work, ['lnco2', 'so2', 'dust', 'water'])

# Save files (UTF-8 + GBK)
out_cols = ['city', 'city_code', 'year', 'treat', 'batch', 'pollution_index_critic_lnco2_3poll']
out = work[out_cols].copy()
out.to_csv('pollution_index_critic_lnco2_3poll_utf8.csv', index=False, encoding='utf-8-sig')

print('saved: pollution_index_critic_lnco2_3poll_utf8.csv')
print('rows =', len(out), ', missing index =', int(out['pollution_index_critic_lnco2_3poll'].isna().sum()))
print('CRITIC weights =', critic_weights.to_dict())

# Prepare full panel for next cells
BASE_PANEL = pd.DataFrame({
    'city': city,
    'city_code': city_code,
    'year': year,
    'treat': treat,
    'batch': batch,
    'y': work['pollution_index_critic_lnco2_3poll'],
    'gdp': pd.to_numeric(df.iloc[:, 10], errors='coerce'),
    'pop': pd.to_numeric(df.iloc[:, 11], errors='coerce'),
    'urban': pd.to_numeric(df.iloc[:, 12], errors='coerce'),
    'fdi': pd.to_numeric(df.iloc[:, 18], errors='coerce'),
    'retail': pd.to_numeric(df.iloc[:, 19], errors='coerce'),
})
BASE_PANEL['ln_gdp_pc'] = np.log((BASE_PANEL['gdp'] / BASE_PANEL['pop']).where((BASE_PANEL['gdp'] > 0) & (BASE_PANEL['pop'] > 0)))
BASE_PANEL['ln_pop'] = np.log(BASE_PANEL['pop'].where(BASE_PANEL['pop'] > 0))
BASE_PANEL['fdi_gdp'] = BASE_PANEL['fdi'] / BASE_PANEL['gdp'].where(BASE_PANEL['gdp'] > 0)
BASE_PANEL['retail_gdp'] = BASE_PANEL['retail'] / BASE_PANEL['gdp'].where(BASE_PANEL['gdp'] > 0)
BASE_PANEL = BASE_PANEL.replace([np.inf, -np.inf], np.nan)


In [ ]:

# 2) Baseline regression table (stepwise controls)
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

if 'BASE_PANEL' not in globals():
    raise RuntimeError('Please run cell 1 first.')

d = BASE_PANEL[['y', 'treat', 'city', 'year', 'ln_gdp_pc', 'ln_pop', 'urban', 'fdi_gdp', 'retail_gdp']].dropna().copy()
d = d[d['treat'].isin([0, 1])]
d['city_fe'] = d['city'].astype(str)
d['year_fe'] = d['year'].astype(int).astype(str)

specs = [
    [],
    ['ln_gdp_pc'],
    ['ln_gdp_pc', 'urban'],
    ['ln_gdp_pc', 'urban', 'ln_pop'],
    ['ln_gdp_pc', 'urban', 'ln_pop', 'fdi_gdp'],
    ['ln_gdp_pc', 'urban', 'ln_pop', 'fdi_gdp', 'retail_gdp'],
]

models = []
for controls in specs:
    rhs = 'treat'
    if controls:
        rhs += ' + ' + ' + '.join(controls)
    f = f'y ~ {rhs} + city_fe + year_fe'
    m = smf.ols(f, data=d).fit(cov_type='cluster', cov_kwds={'groups': d['city']})
    models.append(m)

# export for next cells
BASE_DATA = d.copy()
BEST_CONTROLS = specs[-1].copy()
BEST_MODEL = models[-1]


def stars(p):
    if p < 0.01:
        return '***'
    if p < 0.05:
        return '**'
    if p < 0.10:
        return '*'
    return ''


def coef_se(m, var):
    if var in m.params.index:
        c = m.params[var]
        s = m.bse[var]
        p = m.pvalues[var]
        return f'{c: .4f}{stars(p)}', f'({s: .4f})'
    return '', ''

col_w = 13
name_w = 20
line = '=' * (name_w + (col_w + 1) * 6 + 2)
print(line)
print(' ' * (name_w + 2) + ''.join([f'{f"({i})":>{col_w}} ' for i in range(1, 7)]))
print(line)

rows = [
    ('Policy (treat)', 'treat'),
    ('ln(GDP per capita)', 'ln_gdp_pc'),
    ('Urbanization rate', 'urban'),
    ('ln(Population)', 'ln_pop'),
    ('FDI / GDP', 'fdi_gdp'),
    ('Retail / GDP', 'retail_gdp'),
    ('Constant', 'Intercept'),
]

for label, var in rows:
    coef_line = f'{label:<{name_w}} '
    se_line = ' ' * name_w + ' '
    for m in models:
        c, s = coef_se(m, var)
        coef_line += f'{c:>{col_w}} '
        se_line += f'{s:>{col_w}} '
    print(coef_line)
    print(se_line)

print(f"{'R-squared':<{name_w}} " + ''.join([f'{m.rsquared:>{col_w}.4f} ' for m in models]))
print(f"{'R-squared Adj.':<{name_w}} " + ''.join([f'{m.rsquared_adj:>{col_w}.4f} ' for m in models]))
print(f"{'City FE':<{name_w}} " + ''.join([f"{'Yes':>{col_w}} " for _ in models]))
print(f"{'N':<{name_w}} " + ''.join([f'{int(m.nobs):>{col_w}} ' for m in models]))
print(f"{'Year FE':<{name_w}} " + ''.join([f"{'Yes':>{col_w}} " for _ in models]))
print(line)


In [ ]:

# 3) Parallel trend test (event-study plot)
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt

plt.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei', 'Arial Unicode MS', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

if 'BASE_DATA' not in globals() or 'BEST_CONTROLS' not in globals():
    raise RuntimeError('Please run cell 2 first.')

d = BASE_DATA.copy()
d['city_fe'] = d['city'].astype(str)
d['year_fe'] = d['year'].astype(int).astype(str)
controls = BEST_CONTROLS.copy()

WINDOW = (-3, 4)
BASE_PERIOD = -1
lo, hi = WINDOW

g = d[d['treat'] == 1].groupby('city')['year'].min()
d['g'] = d['city'].map(g)
d['ever'] = d['g'].notna().astype(int)
d['rel'] = d['year'] - d['g']
d['rel_cap'] = d['rel']
d.loc[d['ever'] == 1, 'rel_cap'] = d.loc[d['ever'] == 1, 'rel_cap'].clip(lo, hi)

ks = list(range(lo, hi + 1))
if BASE_PERIOD in ks:
    ks.remove(BASE_PERIOD)


def kname(k):
    return f'ev_m{abs(k)}' if k < 0 else f'ev_p{k}'

term_map = {}
for k in ks:
    nm = kname(k)
    term_map[k] = nm
    d[nm] = ((d['ever'] == 1) & (d['rel_cap'] == k)).astype(int)

rhs = ' + '.join([term_map[k] for k in ks] + controls)
f = f'y ~ {rhs} + city_fe + year_fe'
me = smf.ols(f, data=d).fit(cov_type='cluster', cov_kwds={'groups': d['city']})

lead_terms = [term_map[k] for k in range(lo, 0) if k != BASE_PERIOD and k in term_map]
hyp = ' = 0, '.join(lead_terms) + ' = 0'
wt = me.wald_test(hyp)
print(f'Pre-trend joint test p-value (window {WINDOW}) = {float(wt.pvalue):.6f}')

plot_k = list(range(lo, hi + 1))
coefs, ci_low, ci_high = [], [], []
for k in plot_k:
    if k == BASE_PERIOD:
        coefs.append(0.0); ci_low.append(0.0); ci_high.append(0.0)
    else:
        t = term_map[k]
        b = me.params.get(t, np.nan)
        s = me.bse.get(t, np.nan)
        coefs.append(b); ci_low.append(b - 1.96 * s); ci_high.append(b + 1.96 * s)

fig, ax = plt.subplots(figsize=(8.2, 5.0))
ax.errorbar(
    plot_k,
    coefs,
    yerr=[np.array(coefs) - np.array(ci_low), np.array(ci_high) - np.array(coefs)],
    fmt='o-', color='#333333', ecolor='#666666', elinewidth=1.2, capsize=3, markersize=4
)
ax.axhline(0, color='gray', linestyle='--', linewidth=1)
ax.axvline(0, color='gray', linestyle='--', linewidth=1)
ax.set_xticks(plot_k)
ax.set_xlabel('Relative Time to Policy Start')
ax.set_ylabel('Coefficient')
ax.set_title('CE')
ax.grid(False)
plt.tight_layout()
plt.show()


In [ ]:

# 4) Bootstrap placebo test and plot
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt

plt.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei', 'Arial Unicode MS', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

if 'BASE_DATA' not in globals() or 'BEST_CONTROLS' not in globals() or 'BEST_MODEL' not in globals():
    raise RuntimeError('Please run cell 2 first.')

d = BASE_DATA.copy()
d['city_fe'] = d['city'].astype(str)
d['year_fe'] = d['year'].astype(int).astype(str)
controls = BEST_CONTROLS.copy()

real_beta = float(BEST_MODEL.params['treat'])
real_p = float(BEST_MODEL.pvalues['treat'])
print('Real policy coefficient =', real_beta, ', p =', real_p)

B = 300
rng = np.random.default_rng(20260418)

g_real = d[d['treat'] == 1].groupby('city')['year'].min()
all_cities = np.array(sorted(d['city'].unique()))
num_treated = len(g_real.index)
start_pool = g_real.values

betas, pvals = [], []
formula_fake = 'y ~ fake_treat + ' + ' + '.join(controls) + ' + city_fe + year_fe'

for _ in range(B):
    fake_cities = rng.choice(all_cities, size=num_treated, replace=False)
    fake_starts = rng.choice(start_pool, size=num_treated, replace=True)
    start_map = pd.Series(fake_starts, index=fake_cities)

    start_vec = d['city'].map(start_map)
    dd = d.copy()
    dd['fake_treat'] = ((dd['year'] >= start_vec) & start_vec.notna()).astype(int)

    try:
        m = smf.ols(formula_fake, data=dd).fit(cov_type='cluster', cov_kwds={'groups': dd['city']})
        betas.append(float(m.params['fake_treat']))
        pvals.append(float(m.pvalues['fake_treat']))
    except Exception:
        continue

betas = np.array(betas)
pvals = np.array(pvals)
print(f'Valid placebo runs: {len(betas)} / {B}')
if len(betas) == 0:
    raise RuntimeError('No valid placebo regressions.')

emp_p = np.mean(np.abs(betas) >= abs(real_beta))
print('Empirical placebo p-value =', emp_p)

fig, ax1 = plt.subplots(figsize=(7.6, 5.0))
try:
    from scipy.stats import gaussian_kde
    xs = np.linspace(betas.min() - 0.05, betas.max() + 0.05, 400)
    ys = gaussian_kde(betas)(xs)
    ax1.plot(xs, ys, color='black', lw=1.2, label='Density')
except Exception:
    ax1.hist(betas, bins=30, density=True, color='lightgray', edgecolor='gray', label='Density (Hist)')

ax1.axvline(real_beta, color='gray', linestyle=':', linewidth=1.2)
ax1.set_xlabel('Coefficient')
ax1.set_ylabel('Kernel Density')
ax1.set_title('CE')

ax2 = ax1.twinx()
ax2.scatter(betas, pvals, s=14, facecolors='none', edgecolors='#9a9a9a', alpha=0.85, label='P-value')
ax2.axhline(0.10, color='gray', linestyle='--', linewidth=1.0)
ax2.set_ylabel('P-value')
ax2.set_ylim(-0.02, 1.02)

h1, l1 = ax1.get_legend_handles_labels()
h2, l2 = ax2.get_legend_handles_labels()
ax1.legend(h1 + h2, l1 + l2, frameon=False, loc='upper right')

plt.tight_layout()
plt.show()


In [ ]:

# 5) Robustness check: remove outlier interference (winsorization)
#    Winsorize dependent variable and controls at both tails, then re-estimate DID.
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

if 'BASE_DATA' not in globals() or 'BEST_CONTROLS' not in globals():
    raise RuntimeError('Please run cell 2 first (to create BASE_DATA and BEST_CONTROLS).')

# ---- settings ----
WINSOR_P = 0.01   # 1% two-sided winsorization

# ---- data ----
d0 = BASE_DATA.copy()
controls = BEST_CONTROLS.copy()

if 'city_fe' not in d0.columns:
    d0['city_fe'] = d0['city'].astype(str)
if 'year_fe' not in d0.columns:
    d0['year_fe'] = d0['year'].astype(int).astype(str)

winsor_cols = ['y'] + controls

def winsorize_series(s, p=0.01):
    lo = s.quantile(p)
    hi = s.quantile(1 - p)
    return s.clip(lower=lo, upper=hi), lo, hi

# Apply winsorization
d1 = d0.copy()
bounds = []
for c in winsor_cols:
    x = pd.to_numeric(d1[c], errors='coerce')
    xw, lo, hi = winsorize_series(x, p=WINSOR_P)
    d1[c] = xw
    bounds.append((c, float(lo), float(hi)))

# Same DID specification
formula = 'y ~ treat + ' + ' + '.join(controls) + ' + city_fe + year_fe'

m_before = smf.ols(formula, data=d0).fit(cov_type='cluster', cov_kwds={'groups': d0['city']})
m_after  = smf.ols(formula, data=d1).fit(cov_type='cluster', cov_kwds={'groups': d1['city']})


def star(p):
    if p < 0.01:
        return '***'
    if p < 0.05:
        return '**'
    if p < 0.10:
        return '*'
    return ''

# Print concise comparison
print('=== Outlier-Robustness (Winsorization) ===')
print(f'Winsor level: {WINSOR_P*100:.1f}% each tail')
print('Specification: y ~ treat + controls + city FE + year FE (cluster by city)')

for name, m in [('Before winsor', m_before), ('After winsor', m_after)]:
    b = float(m.params['treat'])
    se = float(m.bse['treat'])
    p = float(m.pvalues['treat'])
    ci = m.conf_int().loc['treat'].tolist()
    print(f'[{name}]')
    print(f'  treat coef = {b:.6f}{star(p)}')
    print(f'  se(cluster city) = {se:.6f}')
    print(f'  p-value = {p:.6f}')
    print(f'  95% CI = [{float(ci[0]):.6f}, {float(ci[1]):.6f}]')
    print(f'  N = {int(m.nobs)}, R2 = {float(m.rsquared):.6f}')

print('\nWinsor bounds (variable, lower, upper):')
for c, lo, hi in bounds:
    print(f'  {c}: [{lo:.6f}, {hi:.6f}]')


In [ ]:

# 6) Improved PSM-DID robustness (city-level pre-policy matching)
#    Steps:
#    (a) define ever-treated city
#    (b) build pre-policy city features (mean + trend of controls)
#    (c) estimate propensity score and do KNN matching with caliper
#    (d) run weighted DID on matched panel sample

import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

if 'BASE_PANEL' not in globals() or 'BEST_CONTROLS' not in globals() or 'BEST_MODEL' not in globals():
    raise RuntimeError('Please run cell 1 and cell 2 first.')

# ---------- settings ----------
K_MATCH = 3             # 1:K nearest-neighbor matching
CALIPER = 0.05          # max |pscore diff| for preferred matches
SEED = 20260418

rng = np.random.default_rng(SEED)
controls = BEST_CONTROLS.copy()  # ['ln_gdp_pc', 'urban', 'ln_pop', 'fdi_gdp', 'retail_gdp']

# ---------- panel data ----------
d = BASE_PANEL[['city', 'year', 'treat', 'y'] + controls].dropna().copy()
d = d[d['treat'].isin([0, 1])]

# first treatment year and ever-treated flag
first_treat = d[d['treat'] == 1].groupby('city')['year'].min()
city_df = pd.DataFrame({'city': sorted(d['city'].unique())})
city_df['g'] = city_df['city'].map(first_treat)
city_df['ever_treated'] = city_df['g'].notna().astype(int)

if city_df['ever_treated'].sum() == 0:
    raise RuntimeError('No treated cities found.')

# global pre-policy cutoff: earliest treated year
global_g0 = int(city_df.loc[city_df['ever_treated'] == 1, 'g'].min())
pre = d[d['year'] < global_g0].copy()

if pre.empty:
    raise RuntimeError('No pre-policy observations found. Check treatment start years.')

# ---------- city-level pre features: mean + trend ----------
def slope_by_city(tmp, ycol):
    out = {}
    for c, g in tmp.groupby('city'):
        gx = g[['year', ycol]].dropna().copy()
        if len(gx) < 2:
            out[c] = np.nan
            continue
        x = gx['year'].to_numpy(dtype=float)
        y = gx[ycol].to_numpy(dtype=float)
        x = x - x.mean()
        denom = np.sum(x ** 2)
        if denom <= 0:
            out[c] = np.nan
        else:
            out[c] = float(np.sum(x * (y - y.mean())) / denom)
    return pd.Series(out)

feat = pd.DataFrame({'city': sorted(pre['city'].unique())})
for v in controls:
    m = pre.groupby('city')[v].mean().rename(f'{v}_mean')
    s = slope_by_city(pre, v).rename(f'{v}_trend')
    feat = feat.merge(m, left_on='city', right_index=True, how='left')
    feat = feat.merge(s, left_on='city', right_index=True, how='left')

city_match = city_df.merge(feat, on='city', how='left')
feature_cols = [c for c in city_match.columns if c.endswith('_mean') or c.endswith('_trend')]
city_match = city_match.dropna(subset=feature_cols).copy()

# keep treated and never-treated only
treated_cities = city_match.loc[city_match['ever_treated'] == 1, 'city'].tolist()
control_cities = city_match.loc[city_match['ever_treated'] == 0, 'city'].tolist()

if len(treated_cities) == 0 or len(control_cities) == 0:
    raise RuntimeError('Need both treated and never-treated cities for PSM.')

# ---------- propensity score ----------
X = city_match[feature_cols].astype(float).copy()
# z-score standardization for numeric stability
X = (X - X.mean()) / X.std(ddof=0).replace(0, np.nan)
X = X.fillna(0.0)
y = city_match['ever_treated'].astype(int).values

use_sklearn = True
try:
    from sklearn.linear_model import LogisticRegression
except Exception:
    use_sklearn = False

if use_sklearn:
    lr = LogisticRegression(max_iter=3000, solver='lbfgs', random_state=SEED)
    lr.fit(X.values, y)
    city_match['pscore'] = lr.predict_proba(X.values)[:, 1]
else:
    import statsmodels.api as sm
    X2 = sm.add_constant(X, has_constant='add')
    glm = sm.GLM(y, X2, family=sm.families.Binomial()).fit()
    city_match['pscore'] = glm.predict(X2)

# ---------- common support ----------
ps_t = city_match.loc[city_match['ever_treated'] == 1, 'pscore']
ps_c = city_match.loc[city_match['ever_treated'] == 0, 'pscore']
lo = max(float(ps_t.min()), float(ps_c.min()))
hi = min(float(ps_t.max()), float(ps_c.max()))
cm = city_match[(city_match['pscore'] >= lo) & (city_match['pscore'] <= hi)].copy()

tr = cm[cm['ever_treated'] == 1].copy()
ct = cm[cm['ever_treated'] == 0].copy()

if tr.empty or ct.empty:
    raise RuntimeError('No overlap after common support trimming.')

# ---------- KNN matching with caliper (with replacement) ----------
control_ps = ct['pscore'].to_numpy()
control_city = ct['city'].to_numpy()

matched_pairs = []
for _, r in tr.iterrows():
    dif = np.abs(control_ps - float(r['pscore']))
    order = np.argsort(dif)

    # prefer within caliper
    in_cal = order[dif[order] <= CALIPER]
    if len(in_cal) >= K_MATCH:
        pick = in_cal[:K_MATCH]
    elif len(in_cal) > 0:
        # if too few within caliper, top-up from nearest outside
        need = K_MATCH - len(in_cal)
        outside = [j for j in order if j not in set(in_cal)]
        pick = np.concatenate([in_cal, np.array(outside[:need], dtype=int)])
    else:
        pick = order[:K_MATCH]

    for j in pick:
        matched_pairs.append((r['city'], control_city[j]))

pairs = pd.DataFrame(matched_pairs, columns=['treated_city', 'control_city'])

# matched control frequency as weight source
ctrl_w = pairs['control_city'].value_counts().rename('match_count').to_frame()
tr_set = set(pairs['treated_city'].unique())
ct_set = set(pairs['control_city'].unique())

matched_cities = sorted(tr_set.union(ct_set))
panel_m = d[d['city'].isin(matched_cities)].copy()

# weights: treated=1, control = match_count / K
panel_m['w'] = 0.0
panel_m.loc[panel_m['city'].isin(tr_set), 'w'] = 1.0
panel_m = panel_m.merge(ctrl_w, left_on='city', right_index=True, how='left')
panel_m.loc[panel_m['city'].isin(ct_set), 'w'] = panel_m.loc[panel_m['city'].isin(ct_set), 'match_count'] / float(K_MATCH)
panel_m['w'] = panel_m['w'].fillna(0.0)
panel_m = panel_m[panel_m['w'] > 0].copy()

panel_m['city_fe'] = panel_m['city'].astype(str)
panel_m['year_fe'] = panel_m['year'].astype(int).astype(str)

# ---------- weighted DID ----------
f = 'y ~ treat + ' + ' + '.join(controls) + ' + city_fe + year_fe'
m_psm = smf.wls(f, data=panel_m, weights=panel_m['w']).fit(
    cov_type='cluster', cov_kwds={'groups': panel_m['city']}
)

# ---------- report ----------
def star(p):
    if p < 0.01:
        return '***'
    if p < 0.05:
        return '**'
    if p < 0.10:
        return '*'
    return ''

print('=== Improved PSM-DID Robustness ===')
print(f'Pre-policy cutoff year: < {global_g0}')
print(f'Matching ratio: 1:{K_MATCH}, caliper={CALIPER}, replacement=True')
print('Controls:', controls)
print('Cities in common support:', len(cm), '| treated:', int((cm['ever_treated']==1).sum()), '| control:', int((cm['ever_treated']==0).sum()))
print('Matched treated cities:', len(tr_set), '| matched control cities:', len(ct_set))

for name, m in [('Baseline DID (full sample)', BEST_MODEL), ('Improved PSM-DID', m_psm)]:
    b = float(m.params['treat'])
    se = float(m.bse['treat'])
    p = float(m.pvalues['treat'])
    ci = m.conf_int().loc['treat'].tolist()
    print(f'[{name}]')
    print(f'  treat coef = {b:.6f}{star(p)}')
    print(f'  se(cluster city) = {se:.6f}')
    print(f'  p-value = {p:.6f}')
    print(f'  95% CI = [{float(ci[0]):.6f}, {float(ci[1]):.6f}]')
    print(f'  N = {int(m.nobs)}, R2 = {float(m.rsquared):.6f}')

# ---------- balance check on city-level features ----------
def weighted_mean(x, w):
    x = np.asarray(x, dtype=float)
    w = np.asarray(w, dtype=float)
    s = np.sum(w)
    return np.nan if s <= 0 else float(np.sum(w * x) / s)

def weighted_var(x, w):
    mu = weighted_mean(x, w)
    if np.isnan(mu):
        return np.nan
    x = np.asarray(x, dtype=float)
    w = np.asarray(w, dtype=float)
    s = np.sum(w)
    return np.nan if s <= 0 else float(np.sum(w * (x - mu) ** 2) / s)

def smd_weighted(x_t, w_t, x_c, w_c):
    mt = weighted_mean(x_t, w_t)
    mc = weighted_mean(x_c, w_c)
    vt = weighted_var(x_t, w_t)
    vc = weighted_var(x_c, w_c)
    sp = np.sqrt((vt + vc) / 2.0)
    if np.isnan(sp) or sp == 0:
        return np.nan
    return float((mt - mc) / sp)

# before matching (common support sample): equal weights within group
cm2 = cm.copy()

# after matching weights at city-level
city_w = pd.DataFrame({'city': sorted(cm['city'].unique())})
city_w['w_t_after'] = city_w['city'].isin(tr_set).astype(float)
city_w = city_w.merge(ctrl_w, left_on='city', right_index=True, how='left')
city_w['w_c_after'] = city_w['match_count'].fillna(0.0) / float(K_MATCH)
city_w = city_w.drop(columns=['match_count'])

cm_after = cm.merge(city_w, on='city', how='left')

print('\nSMD before vs after matching (feature-level, closer to 0 is better):')
for v in controls:
    fv = f'{v}_mean'

    # before: unweighted treated vs control in common support
    xt0 = cm2.loc[cm2['ever_treated'] == 1, fv].to_numpy()
    wt0 = np.ones(len(xt0))
    xc0 = cm2.loc[cm2['ever_treated'] == 0, fv].to_numpy()
    wc0 = np.ones(len(xc0))
    b0 = smd_weighted(xt0, wt0, xc0, wc0)

    # after: treated weight=1, controls weighted by match frequency
    xt1 = cm_after.loc[cm_after['ever_treated'] == 1, fv].to_numpy()
    wt1 = cm_after.loc[cm_after['ever_treated'] == 1, 'w_t_after'].fillna(0.0).to_numpy()
    xc1 = cm_after.loc[cm_after['ever_treated'] == 0, fv].to_numpy()
    wc1 = cm_after.loc[cm_after['ever_treated'] == 0, 'w_c_after'].fillna(0.0).to_numpy()
    b1 = smd_weighted(xt1, wt1, xc1, wc1)

    print(f'  {fv}: before={b0:.4f}, after={b1:.4f}')


In [ ]:

# 7) Robustness: add control-by-time-trend interactions
#    Purpose: absorb differential time trends linked to observed controls.

import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

if 'BASE_DATA' not in globals() or 'BEST_CONTROLS' not in globals():
    raise RuntimeError('Please run cell 2 first (to create BASE_DATA and BEST_CONTROLS).')

d = BASE_DATA.copy()
controls = BEST_CONTROLS.copy()

if 'city_fe' not in d.columns:
    d['city_fe'] = d['city'].astype(str)
if 'year_fe' not in d.columns:
    d['year_fe'] = d['year'].astype(int).astype(str)

# numeric trend (starts from 0)
d['trend'] = d['year'] - d['year'].min()

# baseline (same as model 6)
f0 = 'y ~ treat + ' + ' + '.join(controls) + ' + city_fe + year_fe'
m0 = smf.ols(f0, data=d).fit(cov_type='cluster', cov_kwds={'groups': d['city']})

# add control x trend interactions
inter_terms = ' + '.join([f'{c}:trend' for c in controls])
f1 = f'y ~ treat + ' + ' + '.join(controls) + ' + ' + inter_terms + ' + city_fe + year_fe'
m1 = smf.ols(f1, data=d).fit(cov_type='cluster', cov_kwds={'groups': d['city']})


def star(p):
    if p < 0.01:
        return '***'
    if p < 0.05:
        return '**'
    if p < 0.10:
        return '*'
    return ''

rows = []
for name, m in [('Baseline DID', m0), ('DID + controls?trend', m1)]:
    ci = m.conf_int().loc['treat'].tolist()
    rows.append({
        'Model': name,
        'treat_coef': float(m.params['treat']),
        'treat_se_cluster_city': float(m.bse['treat']),
        'treat_p': float(m.pvalues['treat']),
        'treat_sig': star(float(m.pvalues['treat'])),
        'ci_low': float(ci[0]),
        'ci_high': float(ci[1]),
        'N': int(m.nobs),
        'R2': float(m.rsquared),
    })

res = pd.DataFrame(rows)
print('=== Robustness: Add Control-by-Trend Interactions ===')
print('Controls:', controls)
print(res.to_string(index=False))
